# 1. 消息与提示词模版

## 1.1 认识消息
大模型没有记忆，它的输出只和输入模型的内容有关（上下文）。很多大模型API服务也没有在服务端维护会话历史，是“ 无状态 ”的。**因此，如果应用需要“记住”对话历史，需要在程序中维护消息列表。**

在 LangChain 中，Message（消息）是模型交互的最基本单元。它既代表模型接收到的 输入（Input） ，也代表模型生成的输出（Output） 。
> 每一轮与大模型的对话，都由一条或多条 Message 构成。每个 Message 不仅包含 文字内容 ，还携带描述上下文状态的 元信息（metadata） ，用于保持对话的一致性和可追踪性。比如，模型在多轮交互中理解“谁在说话”、“说了什么”、“这条信息属于哪一轮对话”。

LangChain 在 1.0 中提供了跨模型统一的 Message 标准。无论你使用的是 OpenAI、Anthropic、Gemini 还是本地模型，这一标准都能保持一致的行为。好处：

1. 兼容性强 ：不同模型的消息格式自动对齐。
2. 可扩展性高 ：方便添加多模态内容或自定义字段。
3. 可追踪性好 ：为 LangSmith 等调试工具提供一致的上下文数据结构。



## 1.2 消息的类型

LangChain定义了很多消息类型，通过 role 区分。常用的有四种。
1. 系统消息

也称为系统提示词，用于在对话开始时为模型设定角色、行为准则和上下文背景。它像是给AI助手的一份工作说明书，决定了其回答问题的风格、领域和专业范围。
```json
{"role": "system", "content": "你是个精通编程的软件架构师"}
```

2. 用户消息

也称为用户提示词，在多轮对话中，它表示用户的一次输入。可以包含简单的文本问题，也可以是复杂
的多模态内容（如图片、音频、文档等）。
```json
{"role": "user", "content": "你好啊~"}
```

3. 助手(AI)消息

代表模型的回复，包括生成的文本、工具调用、元数据等。
```json
{"role": "assistant", "content": "我也很高兴认识你"}
{
    "role": "assistant",
    "content": "",
    "tool_calls": [{
    "name": "get_weather",
    "args": {"location": "北京"},
    "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
}]
}
```

4. 工具调用消息

工具调用结果匹配的消息类型。将此消息返回给模型，让模型基于这个结果继续生成回复。在Tools一节详细介绍。
```json
{"role": "tool", "content": "今天天气很好", "tool_call_id":"call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"}
```


问题：为什么使用不同的消息类型？
* 明确角色 ：清晰区分系统提示、用户输入和 AI 回复
* 控制行为 ：通过 SystemMessage 精确控制 AI 的行为
* 对话历史 ：构建完整的多轮对话上下文
* 调试友好 ：更容易追踪和调试对话流程


## 1.3 消息格式
LangChain支持两种消息格式。
### 1.3.1 JSON 格式
```json
{"role": "system", "content": "你是个精通编程的软件架构师"}
```

In [2]:
import time
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

for retries in [0, 2]:
    model = init_chat_model(
        model="deepseek-v4-flash",
        model_provider="deepseek",
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_BASE_URL"),
        max_retries=retries,  # 失败后最多重试几次
    )
    messages = [
        {"role": "system", "content": "你是个善解人意的助手"},
        {"role": "user", "content": "你好"},
    ]
    start = time.time()
    try:
        response = model.invoke(messages)
        print(response.content)
    except Exception as e:
        print(f"max_retries={retries}：{type(e).__name__}: {e}  耗时 {time.time() - start:.2f} 秒")

你好呀！很高兴见到你，有什么我可以帮你的吗？不管是聊聊天、解答问题，还是需要一些建议，我都很乐意帮忙！😊
你好！很高兴见到你。有什么我可以帮你的吗？


### 1.3.2 对象格式

1. 系统消息


In [5]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=1.5,
)

messages = [
    SystemMessage(content="你是个善解人意的助手"),
    HumanMessage(content="帮我讲解一下什么是模型输出时的参数 温度"),
]

response = model.invoke(messages)
print(response)
print(response.content)

content='温度（temperature）可以理解成模型生成时的“随机性旋钮”或“创造力旋钮”。它不改变模型本身的知识，只改变模型在每一步选下一个词时的随机程度。\n\n### 1. 原理：温度怎么起作用\n\n模型生成每个 token 前，会先给所有候选 token 打一个原始分数，叫 **logits**。比如候选词 A、B、C 的分数是 `[2, 1, 0]`。\n\n正常情况下，用 softmax 把这些分数变成概率：\n\n\\[\np_i = \\frac{e^{z_i}}{\\sum_j e^{z_j}}\n\\]\n\n加入温度 \\(T\\) 后，变成：\n\n\\[\np_i = \\frac{e^{z_i / T}}{\\sum_j e^{z_j / T}}\n\\]\n\n其中 \\(T\\) 就是温度。然后模型按这个概率分布随机抽取下一个 token。\n\n### 2. 直观理解\n\n- **低温 \\(T < 1\\)**：概率分布变得更“尖”。高概率词更容易被选中，低概率词几乎没机会。输出更稳定、保守、可复现，但也可能无聊、重复。\n- **\\(T = 1\\)**：不改变模型原始概率分布。\n- **高温 \\(T > 1\\)**：概率分布变得更“平”。低概率词也有机会被选中。输出更多样、有创意，但更容易跑题、胡言乱语、产生幻觉。\n- **\\(T = 0\\)**：通常实现为贪心解码，直接选概率最高的词，每次输出基本固定。\n\n### 3. 一个例子\n\n假设候选词 logits 是 `[2, 1, 0]`：\n\n| 温度 | A 概率 | B 概率 | C 概率 | 效果 |\n|---|---:|---:|---:|---|\n| T = 0.5 | 86.7% | 11.7% | 1.6% | 很保守，几乎总选 A |\n| T = 1 | 66.5% | 24.5% | 9.0% | 原始分布 |\n| T = 2 | 50.6% | 30.7% | 18.6% | 更随机，C 也有机会 |\n\n当 \\(T \\to 0\\)，最高分词的趋近 100%；当 \\(T \\to \\infty\\)，所有词趋近均匀分布。\n\n### 4. 和 top-p、top-k 的区别\n\n- **温度**：

## 1.4 消息对象字段说明
此处仅说明常用字段，完整字段列表查阅官方手册或阅读源码。
 SystemMessage、HumanMessage、AIMessage、ToolMessage

这几个类都继承自 `BaseMessage`，继承关系如下：

```
BaseMessage（公共字段）
├── SystemMessage        系统消息
├── HumanMessage         用户消息
├── AIMessage            AI 消息     + tool_calls、invalid_tool_calls、usage_metadata
│   └── AIMessageChunk   流式输出的消息块 + tool_call_chunks、chunk_position
└── ToolMessage          工具消息    + tool_call_id、artifact、status
```

**公共字段（所有消息都有）**

| 字段 | 类型 | 作用 |
|---|---|---|
| `content` | `str` 或 `list` | 消息内容。纯文本时是字符串；多模态（文字 + 图片等）时是内容块列表（见 1.4.2） |
| `type` | `str` | 消息类型，由类决定，不需要手动设置：`system` / `human` / `ai` / `tool` |
| `name` | `str` | 可选，发送者的名字，多人对话时用来区分不同用户 |
| `id` | `str` | 可选，消息的唯一 ID。模型返回的 AIMessage 会自动生成（`lc_run--…`） |
| `additional_kwargs` | `dict` | 厂商返回的非标准字段，如 DeepSeek 的思考过程 `reasoning_content` |
| `response_metadata` | `dict` | 响应元数据，主要出现在模型返回的 AIMessage 中（模型名、结束原因、token 用量等） |

**常用属性和方法**

| 名称 | 作用 |
|---|---|
| `.text` | 取出文本内容。`content` 是列表时，自动拼接其中的文本块 |
| `.content_blocks` | 把 `content` 转换成 LangChain 统一格式的内容块列表 |
| `.pretty_print()` | 格式化打印消息（`04-model-invoke.ipynb` 中用过） |

> 导入路径：LangChain 1.x 推荐写 `from langchain.messages import HumanMessage`。它和 `langchain_core.messages` 中的是**同一批类**，两种写法都可以。

下面用代码查看每个类的字段，找出各自特有的字段：

In [1]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.messages import BaseMessage

common = list(BaseMessage.model_fields)
print("公共字段:", common, "\n")
for cls in [SystemMessage, HumanMessage, AIMessage, ToolMessage]:
    own = [f for f in cls.model_fields if f not in common]  # 去掉公共字段，剩下的就是特有字段
    print(f"{cls.__name__:<14} type={cls.model_fields['type'].default!r:<10} 特有字段: {own or '无'}")

公共字段: ['content', 'additional_kwargs', 'response_metadata', 'type', 'name', 'id'] 

SystemMessage  type='system'   特有字段: 无
HumanMessage   type='human'    特有字段: 无
AIMessage      type='ai'       特有字段: ['tool_calls', 'invalid_tool_calls', 'usage_metadata']
ToolMessage    type='tool'     特有字段: ['tool_call_id', 'artifact', 'status']


看看公共字段的用法。创建消息时，`content` 可以作为第一个位置参数直接传入，`HumanMessage("你好")` 等价于 `HumanMessage(content="你好")`：

In [2]:
from langchain.messages import HumanMessage

msg = HumanMessage("你好，我想咨询一下退货流程", name="xiaoming", id="msg-001")
print(msg)
print("type:", msg.type, "| name:", msg.name, "| id:", msg.id)
print("text:", msg.text)
msg.pretty_print()

content='你好，我想咨询一下退货流程' additional_kwargs={} response_metadata={} name='xiaoming' id='msg-001'
type: human | name: xiaoming | id: msg-001
text: 你好，我想咨询一下退货流程
================================ Human Message =================================
Name: xiaoming

你好，我想咨询一下退货流程


### 1.4.1 SystemMessage（系统消息）

**作用**：为模型设定角色、行为准则和回答风格，相当于给模型的一份"工作说明书"。对应 JSON 格式中的 `"role": "system"`，`type` 为 `system`。

**特有属性**：无，只有公共字段，最常用的就是 `content`。

使用要点：

- 一般放在消息列表的**第一条**，对整个对话生效；
- 同一个问题，换一个系统消息，回答的角度和风格就完全不同：

In [3]:
from langchain.messages import SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

question = HumanMessage("为什么天空是蓝色的？")
system_prompts = [
    "你是一位严谨的物理学教授，回答要专业准确，不超过 100 字",
    "你是一位给小学生讲课的老师，回答要通俗有趣，不超过 50 字",
]
for prompt in system_prompts:
    response = model.invoke([SystemMessage(prompt), question])
    print(f"【{prompt[:12]}…】\n{response.content}\n")

【你是一位严谨的物理学教授…】
瑞利散射所致。太阳光穿过大气时，波长较短的蓝紫光被空气分子散射的强度远大于红光，与波长四次方成反比。蓝光散射后布满天空，故呈蓝色；紫光多被吸收，人眼对蓝光也更敏感。



【你是一位给小学生讲课的老…】
太阳光里有七种颜色，蓝光最调皮，爱被空气里的小颗粒撞来撞去，撒得满天都是，所以天空看起来蓝蓝的！



### 1.4.2 HumanMessage（用户消息）

**作用**：代表用户的一次输入。对应 JSON 格式中的 `"role": "user"`，`type` 为 `human`。

**特有属性**：无。有两个公共字段在 HumanMessage 中比较常用：

1. **`name`**：多个用户参与同一个对话时，用来区分是谁说的（上面的例子中设置过）；
2. **`content` 可以是列表，实现多模态输入**：一条消息里同时包含文字、图片、音频、文件等。

列表中的每一项叫一个**内容块**，LangChain 统一的格式如下：

| `type` | 用途 | 数据来源（三选一） |
|---|---|---|
| `text` | 文字 | `text` 字段 |
| `image` | 图片 | `url`（图片地址）、`base64`（图片数据）或 `file_id`（厂商文件 ID），配合 `mime_type`（如 `image/png`） |
| `audio` | 音频 | 同上 |
| `file` | 文件（如 PDF） | 同上 |

LangChain 会把统一格式转换成各厂商要求的格式（比如 OpenAI 的 `image_url`）。但模型能不能看懂，取决于它是否支持这类输入，也就是第 2 章 `07-profile-initparams-config.ipynb` 中模型画像的 `image_inputs` 等字段：DeepSeek V4 Flash 支持图片，V4 Pro 不支持。

下面用代码生成一张纯红色的图片（不需要准备图片文件），分别发给这两个模型：

In [4]:
import base64
import struct
import zlib
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

def solid_color_png(width, height, rgb):
    """生成一张纯色 PNG 图片，返回图片的二进制数据（只用于演示，不需要看懂）"""
    rows = b"".join(b"\x00" + bytes(rgb) * width for _ in range(height))
    def chunk(tag, data):
        return struct.pack(">I", len(data)) + tag + data + struct.pack(">I", zlib.crc32(tag + data) & 0xFFFFFFFF)
    header = struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0)
    return b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", header) + chunk(b"IDAT", zlib.compress(rows)) + chunk(b"IEND", b"")

image_base64 = base64.b64encode(solid_color_png(64, 64, (220, 30, 30))).decode()

message = HumanMessage(content=[
    {"type": "text", "text": "这张图片是什么颜色？只回答颜色"},
    {"type": "image", "base64": image_base64, "mime_type": "image/png"},
])
print("内容块类型:", [block["type"] for block in message.content_blocks])

for name in ["deepseek-v4-flash", "deepseek-v4-pro"]:
    model = init_chat_model(
        model=name,
        model_provider="deepseek",
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_BASE_URL"),
        extra_body={"thinking": {"type": "disabled"}},
    )
    response = model.invoke([message])
    print(f"{name}：{response.content}（输入 token 数：{response.usage_metadata['input_tokens']}）")

内容块类型: ['text', 'image']


deepseek-v4-flash：红色（输入 token 数：197）


deepseek-v4-pro：抱歉，我无法查看这张图片。（输入 token 数：18）


V4 Flash 正确识别出了颜色；V4 Pro 回答看不到图片，而且输入 token 数明显更少，说明图片根本没有被模型处理。**所以使用多模态输入前，先确认模型是否支持**（可以检查 `model.profile.get("image_inputs")`）。

### 1.4.3 AIMessage（AI 消息）

**作用**：代表模型的回复。对应 JSON 格式中的 `"role": "assistant"`，`type` 为 `ai`。

AIMessage 有两种来源：

1. **模型返回**：`invoke()` 的返回值，字段最全，`04-model-invoke.ipynb` 的 4.1.3 节已经逐个字段说明过；
2. **手动创建**：构造对话历史时，把之前的回答写成 AIMessage（如 04 笔记中的 `AIMessage(content="8")`）；或者手写几组"问题 → 回答"作为示例，让模型模仿（few-shot，见下面的例子）。

**特有属性**：

| 字段 | 作用 |
|---|---|
| `tool_calls` | 模型要求调用的工具列表。每一项包含 `name`（工具名）、`args`（参数字典）、`id`（这次调用的 ID）、`type`（固定为 `tool_call`） |
| `invalid_tool_calls` | 解析失败的工具调用，比如模型给出的参数不是合法的 JSON |
| `usage_metadata` | LangChain 统一格式的 token 用量。只有模型返回的消息才有，手动创建的为 `None` |

工具调用会在 Tools 一节详细介绍，这里先看看 `tool_calls` 的结构。模型需要调用工具时，`content` 通常为空，要调用的工具放在 `tool_calls` 中：

In [5]:
from langchain.messages import AIMessage

ai_msg = AIMessage(
    content="",
    tool_calls=[{"name": "get_weather", "args": {"city": "北京"}, "id": "call_001"}],
)
print("tool_calls:", ai_msg.tool_calls)            # LangChain 自动补上了 type='tool_call'
print("content_blocks:", ai_msg.content_blocks)  # 统一格式中，工具调用也是一种内容块
print("usage_metadata:", ai_msg.usage_metadata)  # 手动创建的消息没有 token 用量

tool_calls: [{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_001', 'type': 'tool_call'}]
content_blocks: [{'type': 'tool_call', 'id': 'call_001', 'name': 'get_weather', 'args': {'city': '北京'}}]
usage_metadata: None


**手动创建 AIMessage 的典型用法：few-shot 示例**

在消息列表中放几组"用户问题 → 期望的回答"，模型就会模仿这些回答的格式和风格。这比在系统消息里用文字描述格式更直观：

In [6]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

messages = [
    SystemMessage("判断用户评论的情感倾向"),
    HumanMessage("这家店的菜太好吃了，下次还来！"),
    AIMessage("😊 积极"),                  # 手写的"示例回答"
    HumanMessage("等了一个小时还没上菜，太失望了"),
    AIMessage("😠 消极"),
    HumanMessage("环境一般，价格还算合理"),  # 真正要判断的评论
]
print(model.invoke(messages).content)

😐 中性


模型模仿了示例的格式，只输出了"表情 + 情感"，没有多余的解释。

**流式输出中的 AIMessageChunk**

`05-model-stream-batch.ipynb` 中用过的 `stream()`，每次返回的是 **`AIMessageChunk`**（AIMessage 的"碎片"）。它继承自 AIMessage，额外有两个字段：

| 字段 | 作用 |
|---|---|
| `tool_call_chunks` | 工具调用的片段（流式输出时，工具参数也是一段一段返回的） |
| `chunk_position` | 为 `"last"` 时表示这是最后一块 |

多个 AIMessageChunk 可以直接用 `+` 拼接，得到完整的消息：

In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

full = None
for i, chunk in enumerate(model.stream("用一句话介绍北京")):
    if i < 3:
        print(f"第 {i + 1} 块：{type(chunk).__name__}，content={chunk.content!r}")
    full = chunk if full is None else full + chunk  # 用 + 把碎片拼起来

print("拼接后：", type(full).__name__, "| chunk_position:", full.chunk_position)
print("完整内容：", full.content)
print("token 用量：", full.usage_metadata)

第 1 块：AIMessageChunk，content=''


第 2 块：AIMessageChunk，content='北京'
第 3 块：AIMessageChunk，content='是中国'


拼接后： AIMessageChunk | chunk_position: last
完整内容： 北京是中国首都，一座融合了三千年建城史与现代化发展的世界级历史文化名城。
token 用量： {'input_tokens': 8, 'output_tokens': 19, 'total_tokens': 27, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


### 1.4.4 ToolMessage（工具消息）

**作用**：把工具的执行结果交给模型，模型再根据结果继续回答。对应 JSON 格式中的 `"role": "tool"`，`type` 为 `tool`。

**特有属性**：

| 字段 | 是否必填 | 作用 | 会发给模型吗 |
|---|---|---|---|
| `tool_call_id` | **必填** | 对应 AIMessage 的 `tool_calls` 中的 `id`，说明这是哪一次调用的结果 | 会 |
| `status` | 可选 | 工具是否执行成功：`"success"`（默认）或 `"error"` | 不会 |
| `artifact` | 可选 | 工具产生的原始数据（如完整的查询结果），留给程序自己使用 | 不会 |

一次完整的工具调用由 4 条消息组成：

```
HumanMessage（提问）→ AIMessage（tool_calls：要调用哪个工具）→ ToolMessage（工具结果，tool_call_id 对应）→ AIMessage（最终回答）
```

下面手动模拟这个过程（真实场景中第二条消息由模型生成，Tools 一节会学到）：

> ⚠️ 手动编写带 `tool_calls` 的 AIMessage 时，要**关闭 DeepSeek 的思考模式**。否则 DeepSeek 会报错，要求把这条消息的思考内容（`reasoning_content`）一起传回去，而手写的消息没有思考内容。

In [8]:
from langchain.messages import HumanMessage, AIMessage, ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

messages = [
    HumanMessage("北京今天天气怎么样？"),
    AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"city": "北京"}, "id": "call_001"}]),
    ToolMessage(content="北京：晴，25℃，东南风3级", tool_call_id="call_001"),  # 假装是天气工具返回的结果
]
response = model.invoke(messages)
print(response.content)

北京今天天气晴朗，气温25℃，东南风3级，体感比较舒适。适合外出，注意防晒即可。


模型根据工具返回的数据给出了回答。

**规则：每个 tool_call 都必须有对应的 ToolMessage**。缺少 ToolMessage，或者 `tool_call_id` 对不上，API 都会直接报错：

In [9]:
try:
    model.invoke(messages[:2])  # 去掉 ToolMessage
except Exception as e:
    print(type(e).__name__, str(e)[:170])

OpenAIInvalidRequestError Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. (insufficient tool me


**`status` 和 `artifact` 不会发给模型**。用 `convert_to_openai_messages()` 看看消息实际发送时的样子（1.5.3 节会介绍这个函数）：

In [10]:
from langchain.messages import ToolMessage
from langchain_core.messages import convert_to_openai_messages

tool_msg = ToolMessage(
    content="天气服务超时，查询失败",
    tool_call_id="call_001",
    status="error",                                     # 标记执行失败
    artifact={"http_status": 504, "elapsed_ms": 30000},  # 原始数据，留给程序使用
)
print("消息对象中:", tool_msg.status, tool_msg.artifact)
print("实际发给模型:", convert_to_openai_messages([tool_msg]))

消息对象中: error {'http_status': 504, 'elapsed_ms': 30000}
实际发给模型: [{'role': 'tool', 'tool_call_id': 'call_001', 'content': '天气服务超时，查询失败'}]


发给模型的只有 `content` 和 `tool_call_id`。所以：

- **想让模型知道工具执行失败了，要把原因写进 `content`**。`status` 是给程序看的标记，比如 Agent 框架会在工具报错时把它设为 `"error"`；
- 工具返回的数据很大时（比如一整张表格），可以只把摘要放进 `content` 发给模型，节省 token；完整数据放进 `artifact`，留给程序后续使用。

### 1.4.5 其他消息类型（了解即可）

| 类 | 作用 |
|---|---|
| `ChatMessage` | 自定义角色的消息，通过 `role` 字段指定角色名，很少使用 |
| `RemoveMessage` | 在 LangGraph 中用来删除对话历史中的某条消息 |
| `FunctionMessage` | 旧版"函数调用"的结果消息，已被 ToolMessage 取代 |
| `HumanMessageChunk` 等 | 各类消息的"碎片"版本，用于流式输出 |

## 1.5 消息的常用操作

### 1.5.1 维护对话历史

1.1 节提到：大模型没有记忆，要"记住"对话，就得在程序中维护消息列表。做法是：每轮把用户的问题（HumanMessage）和模型的回答（AIMessage）都追加到列表中，下一轮把整个列表发给模型：

In [11]:
from langchain.messages import SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

history = [SystemMessage("你是一个友好的助手，回答尽量简短")]
for question in ["你好，我叫小明", "我最喜欢的颜色是蓝色", "我叫什么名字？喜欢什么颜色？"]:
    history.append(HumanMessage(question))  # 1. 追加用户的问题
    response = model.invoke(history)        # 2. 把完整的历史发给模型
    history.append(response)                # 3. 追加模型的回答（response 本身就是 AIMessage）
    print(f"用户：{question}\nAI：{response.content}\n")

print("历史中共有", len(history), "条消息：", [m.type for m in history])

用户：你好，我叫小明
AI：你好，小明！很高兴认识你。



用户：我最喜欢的颜色是蓝色
AI：蓝色很好！天空和大海都是蓝色的。



用户：我叫什么名字？喜欢什么颜色？
AI：你叫小明，喜欢蓝色。

历史中共有 7 条消息： ['system', 'human', 'ai', 'human', 'ai', 'human', 'ai']


第三轮模型能答出名字和颜色，因为前两轮的内容都在发给它的消息列表里。

### 1.5.2 裁剪对话历史

对话越长，每次发送的 token 越多，费用越高，还可能超过模型的上下文窗口。`trim_messages()` 可以按 token 数裁剪历史，只保留最近的几轮。常用参数：

| 参数 | 作用 |
|---|---|
| `max_tokens` | 裁剪后最多保留多少 token |
| `token_counter` | 如何计算 token 数。`count_tokens_approximately` 是按字符数估算的快速方法；也可以传入模型对象，按模型的算法计算 |
| `strategy="last"` | 保留最后（最新）的消息 |
| `include_system=True` | 始终保留系统消息 |
| `start_on="human"` | 裁剪后的历史以用户消息开头，避免开头是一条孤立的 AI 回答 |

下面构造 5 轮对话，裁剪到 80 个 token 以内（不调用模型）：

In [12]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately

history = [SystemMessage("你是一个友好的助手")]
for i in range(1, 6):
    history.append(HumanMessage(f"第{i}个问题：" + "内容" * 10))
    history.append(AIMessage(f"第{i}个回答：" + "内容" * 10))

trimmed = trim_messages(
    history,
    max_tokens=80,
    token_counter=count_tokens_approximately,
    strategy="last",
    include_system=True,
    start_on="human",
)
print(f"裁剪前：{len(history)} 条，约 {count_tokens_approximately(history)} token")
print(f"裁剪后：{len(trimmed)} 条，约 {count_tokens_approximately(trimmed)} token")
for m in trimmed:
    print("   ", type(m).__name__, m.content[:8])

裁剪前：11 条，约 122 token
裁剪后：7 条，约 76 token
    SystemMessage 你是一个友好的助
    HumanMessage 第3个问题：内容
    AIMessage 第3个回答：内容
    HumanMessage 第4个问题：内容
    AIMessage 第4个回答：内容
    HumanMessage 第5个问题：内容
    AIMessage 第5个回答：内容


系统消息保留了，前两轮对话被丢掉，保留了最近 3 轮。

### 1.5.3 格式转换与保存

`04-model-invoke.ipynb` 中提到，消息对象格式的缺点是"不方便序列化"。LangChain 提供了几个转换函数来解决：

| 函数 | 作用 |
|---|---|
| `convert_to_messages()` | 字典、元组、字符串 → 消息对象 |
| `convert_to_openai_messages()` | 消息对象 → OpenAI 格式的字典（就是 1.3.1 节的 JSON 格式） |
| `messages_to_dict()` / `messages_from_dict()` | 消息对象 ↔ 完整保留所有字段的字典，可以存成 JSON 文件或存进数据库 |

In [13]:
import json
from langchain_core.messages import convert_to_messages, convert_to_openai_messages, messages_to_dict, messages_from_dict

# 1. 字典、元组、字符串 → 消息对象
messages = convert_to_messages([
    {"role": "system", "content": "你是天气助手"},
    ("human", "北京天气怎么样？"),
    "上海呢？",  # 单独的字符串会被当作 HumanMessage
])
print("1.", [type(m).__name__ for m in messages])

# 2. 消息对象 → OpenAI 格式的字典
print("2.", convert_to_openai_messages(messages))

# 3. 保存对话历史：消息对象 → JSON 字符串 → 消息对象
saved = json.dumps(messages_to_dict(messages), ensure_ascii=False)  # 可以写入文件或数据库
restored = messages_from_dict(json.loads(saved))
print("3. JSON 长度:", len(saved), "| 恢复后与原来相同:", restored == messages)

1. ['SystemMessage', 'HumanMessage', 'HumanMessage']
2. [{'role': 'system', 'content': '你是天气助手'}, {'role': 'user', 'content': '北京天气怎么样？'}, {'role': 'user', 'content': '上海呢？'}]
3. JSON 长度: 431 | 恢复后与原来相同: True


## 1.6 总结

**四种常用消息对比**

| 消息类 | `type` | JSON 中的 `role` | 作用 | 特有属性 |
|---|---|---|---|---|
| `SystemMessage` | `system` | `system` | 设定模型的角色、规则和风格，一般放在第一条 | 无 |
| `HumanMessage` | `human` | `user` | 用户的输入，`content` 可以是多模态内容块列表 | 无 |
| `AIMessage` | `ai` | `assistant` | 模型的回复；也可以手动创建，用于对话历史和 few-shot 示例 | `tool_calls`、`invalid_tool_calls`、`usage_metadata` |
| `ToolMessage` | `tool` | `tool` | 把工具的执行结果交给模型 | `tool_call_id`（必填）、`status`、`artifact` |

所有消息都有公共字段：`content`、`type`、`name`、`id`、`additional_kwargs`、`response_metadata`。流式输出返回的是 `AIMessageChunk`，可以用 `+` 拼接。

**要点回顾**

1. 大模型没有记忆，对话历史要在程序中用消息列表维护，每轮追加 HumanMessage 和 AIMessage；历史太长时用 `trim_messages()` 裁剪。
2. SystemMessage 决定回答的风格，同一个问题换一个系统消息，回答完全不同。
3. HumanMessage 的 `content` 写成内容块列表即可发送图片等多模态内容，但前提是模型支持（看模型画像）。
4. 手动创建 AIMessage 可以做 few-shot 示例，让模型模仿回答格式。
5. 每个 `tool_call` 都必须有 `tool_call_id` 对应的 ToolMessage，否则 API 报错。`status`、`artifact` 不会发给模型，失败原因要写在 `content` 中。
6. 手动编写带 `tool_calls` 的 AIMessage 时，要关闭 DeepSeek 的思考模式。
7. 字典格式和对象格式可以用 `convert_to_messages()` / `convert_to_openai_messages()` 互相转换，用 `messages_to_dict()` / `messages_from_dict()` 保存和恢复对话历史。